In [ ]:
"""
CONSOLIDATION: combine every resolution pass built so far into one
final entity mapping, applied to the main triples file.

Three passes, applied in this order (each only touches what the
previous pass left untouched):
  1. Per-paper definitions        (step0_abbreviation_expansion.py, "detected_definitions" sheet)
  2. Static global dictionary     (same constants as step0, standard reagents/methods only)
  3. Verified full-article lookup (step0b_full_article_resolution.py, "verified_resolutions.xlsx")

Deliberately EXCLUDED from auto-application:
  - unverified_needs_review.xlsx  (LLM answer, quote not confirmed, not trustworthy yet)
  - skipped_id_style_codes.xlsx   (genotype/sample IDs, correctly left as-is, nothing to apply)

Two outputs:
  - expanded_triples.xlsx   : your main triples file with expanded_source/
    expanded_target columns added. Single sheet, this is a flat data
    table, no multi-sheet issue, and it's the file to feed into
    tier1/tier2/tier3 next.
  - consolidation_audit.xlsx : a multi-sheet workbook for YOUR OWN
    review in Excel (not meant for re-upload), with named sheets
    breaking down what each pass contributed and what's still
    unresolved after all three.

Designed for Jupyter/Colab execution. No __main__ guard.
"""

import re
import pandas as pd

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PART-3-PREPROCESSING-TRIPLICATE\phase2_terra_full_triples_categorized.xlsx"  # swap to full corpus file when ready
SOURCE_COL = "source"
TARGET_COL = "target"
DOI_COL = "doi"

ABBREV_REVIEW_PATH = "abbreviation_expansion_review.xlsx"  # from step0, needs "detected_definitions" sheet
FULL_ARTICLE_RESOLUTION_PATH = "full_article_resolution_review.xlsx"  # from step0b, sheets: verified_resolutions, unverified_needs_review

OUTPUT_TRIPLES_XLSX = "expanded_triples.xlsx"
OUTPUT_AUDIT_XLSX = "consolidation_audit.xlsx"

# Corrections from your manual review of unverified_needs_review.xlsx.
# Every other non-null row in that sheet you confirmed as correct and
# is used as-is; these 5 pairs specifically had a wrong LLM answer and
# your corrected value overrides it. (doi, code) -> corrected full_form.
HUMAN_CORRECTIONS = {
    ("10-1016_j-foodchem-2013-12-108", "ISOPRO"): "commercial soy protein concentrate",
    ("10-1016_j-foodhyd-2019-105416", "PHPC"): "Pinus halepensis protein concentrate",
    ("10-1016_j-foodres-2024-115267", "PAGE"): "polyacrylamide gel electrophoresis",
    ("10-1016_j-indcrop-2024-119481", "II"): "Emulsifying Activity Index (EAI)",
    ("10-1016_j-indcrop-2024-119481", "III"): "Emulsion Stability Index (ESI)",
}

# Confirmed NOT abbreviations, commercial product/sample codes with no
# real name to resolve to (same category as genotype IDs like SP-10,
# just a different naming convention). Confirmed against the actual
# paper: EPI 80, E86 HV, E86 LS are product names from AOT and Emsland
# Group, not shorthand for a longer descriptive phrase. Excluded here
# regardless of what any upstream resolution file says, this overrides
# stale "verified" rows generated before the step0b table-row/no-real-
# name checks were added.
CONFIRMED_NOT_ABBREVIATIONS = {
    ("10-1016_j-foodhyd-2024-110996", "EPI"),
    ("10-1016_j-foodhyd-2024-110996", "HV"),
    ("10-1016_j-foodhyd-2024-110996", "LS"),
}

# same static dictionary as step0_abbreviation_expansion.py, standard
# chemistry reagents/methods only, never material/sample names (PPI
# means different things in different papers, confirmed earlier, so
# it must stay out of any global list)
GLOBAL_ABBREVIATIONS = {
    "KOH": "potassium hydroxide", "NAOH": "sodium hydroxide", "HCL": "hydrochloric acid",
    "MDA": "malondialdehyde", "AAPH": "2,2'-azobis(2-amidinopropane) dihydrochloride",
    "DPPH": "2,2-diphenyl-1-picrylhydrazyl",
    "ABTS": "2,2'-azino-bis(3-ethylbenzothiazoline-6-sulfonic acid)",
    "SDS": "sodium dodecyl sulfate", "FTIR": "fourier-transform infrared spectroscopy",
    "SEM": "scanning electron microscopy", "XRD": "x-ray diffraction",
    "NMR": "nuclear magnetic resonance", "DSC": "differential scanning calorimetry",
    "ITC": "isothermal titration calorimetry", "CD": "circular dichroism",
    "ANN": "artificial neural network", "GA": "genetic algorithm",
    "PBS": "phosphate-buffered saline", "EDTA": "ethylenediaminetetraacetic acid",
    "BSA": "bovine serum albumin",
}

# ---------------------------------------------------------------
# LOAD ALL THREE LOOKUPS
# ---------------------------------------------------------------
def load_triples(path):
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    elif path.endswith(".xlsx"):
        return pd.read_excel(path)
    elif path.endswith(".csv"):
        return pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")


df = load_triples(TRIPLES_PATH)
print(f"Loaded {len(df)} triples")

# pass 1: per-paper definitions
per_paper_df = pd.read_excel(ABBREV_REVIEW_PATH, sheet_name="detected_definitions")
per_paper_lookup = {}
for _, row in per_paper_df.iterrows():
    per_paper_lookup.setdefault(row[DOI_COL], {})[row["abbreviation"]] = row["full_phrase"]
print(f"Pass 1 (per-paper definitions): {len(per_paper_df)} entries across {per_paper_df[DOI_COL].nunique()} papers")

# pass 2: static global dictionary (already defined above, no file needed)
print(f"Pass 2 (static global dictionary): {len(GLOBAL_ABBREVIATIONS)} entries")

# pass 3: verified full-article resolutions
verified_df = pd.read_excel(FULL_ARTICLE_RESOLUTION_PATH, sheet_name="verified_resolutions")
verified_lookup = {}
n_excluded_pass3 = 0
for _, row in verified_df.iterrows():
    if (row[DOI_COL], row["code"]) in CONFIRMED_NOT_ABBREVIATIONS:
        n_excluded_pass3 += 1
        continue
    verified_lookup.setdefault(row[DOI_COL], {})[row["code"]] = row["full_form"]
print(f"Pass 3 (verified full-article resolutions): {len(verified_df) - n_excluded_pass3} entries "
      f"across {verified_df[DOI_COL].nunique()} papers ({n_excluded_pass3} excluded as confirmed non-abbreviations)")

# pass 4: human-reviewed unverified rows. The quote-verification check
# in step0b flagged these because it couldn't confirm the LLM's exact
# quote, not because the answer was necessarily wrong, you read all 38
# and confirmed all but 5 nulls were correct as-is, with 5 corrected.
unverified_df = pd.read_excel(FULL_ARTICLE_RESOLUTION_PATH, sheet_name="unverified_needs_review")
human_reviewed_df = unverified_df[unverified_df["full_form"].notna()].copy()
human_reviewed_df = human_reviewed_df[
    ~human_reviewed_df.apply(lambda r: (r[DOI_COL], r["code"]) in CONFIRMED_NOT_ABBREVIATIONS, axis=1)
]
human_reviewed_df["full_form"] = human_reviewed_df.apply(
    lambda r: HUMAN_CORRECTIONS.get((r[DOI_COL], r["code"]), r["full_form"]), axis=1
)
human_reviewed_lookup = {}
for _, row in human_reviewed_df.iterrows():
    human_reviewed_lookup.setdefault(row[DOI_COL], {})[row["code"]] = row["full_form"]
n_corrected = sum(1 for _, r in human_reviewed_df.iterrows()
                   if (r[DOI_COL], r["code"]) in HUMAN_CORRECTIONS)
print(f"Pass 4 (human-reviewed unverified rows): {len(human_reviewed_df)} entries "
      f"across {human_reviewed_df[DOI_COL].nunique()} papers ({n_corrected} corrected by you, "
      f"{len(unverified_df) - len(human_reviewed_df)} nulls excluded)")

# ---------------------------------------------------------------
# APPLY ALL THREE PASSES, IN ORDER, TRACKING WHICH PASS RESOLVED WHAT
# ---------------------------------------------------------------
def expand_entity(entity, doi):
    if pd.isna(entity):
        return entity, []
    text = str(entity)
    resolved_by = []

    for abbr, full_phrase in per_paper_lookup.get(doi, {}).items():
        pattern = re.compile(rf"\b{re.escape(abbr)}\b")
        if pattern.search(text):
            text = pattern.sub(full_phrase, text)
            resolved_by.append("per_paper")

    for abbr, full_phrase in GLOBAL_ABBREVIATIONS.items():
        pattern = re.compile(rf"\b{re.escape(abbr)}\b", re.IGNORECASE)
        if pattern.search(text):
            text = pattern.sub(full_phrase, text)
            resolved_by.append("global_dict")

    for code, full_form in verified_lookup.get(doi, {}).items():
        if pd.isna(full_form):
            continue
        pattern = re.compile(rf"\b{re.escape(str(code))}\b")
        if pattern.search(text):
            text = pattern.sub(str(full_form), text)
            resolved_by.append("verified_full_article")

    for code, full_form in human_reviewed_lookup.get(doi, {}).items():
        if pd.isna(full_form):
            continue
        pattern = re.compile(rf"\b{re.escape(str(code))}\b")
        if pattern.search(text):
            text = pattern.sub(str(full_form), text)
            resolved_by.append("human_reviewed")

    return text, resolved_by


expanded_sources, expanded_targets = [], []
source_resolved_by, target_resolved_by = [], []

for _, row in df.iterrows():
    exp_src, src_by = expand_entity(row[SOURCE_COL], row[DOI_COL])
    exp_tgt, tgt_by = expand_entity(row[TARGET_COL], row[DOI_COL])
    expanded_sources.append(exp_src)
    expanded_targets.append(exp_tgt)
    source_resolved_by.append("; ".join(sorted(set(src_by))))
    target_resolved_by.append("; ".join(sorted(set(tgt_by))))

df["expanded_source"] = expanded_sources
df["expanded_target"] = expanded_targets
df["source_resolved_by"] = source_resolved_by
df["target_resolved_by"] = target_resolved_by

n_source_changed = (df["expanded_source"] != df[SOURCE_COL].astype(str)).sum()
n_target_changed = (df["expanded_target"] != df[TARGET_COL].astype(str)).sum()
print(f"\nRows with source expanded: {n_source_changed}")
print(f"Rows with target expanded: {n_target_changed}")

# ---------------------------------------------------------------
# SAVE MAIN OUTPUT: single sheet, this is the flat data table to
# feed into tier1_family_tagging.py / tier2 / tier3 next
# ---------------------------------------------------------------
df.to_excel(OUTPUT_TRIPLES_XLSX, index=False)
print(f"\nSaved expanded triples to {OUTPUT_TRIPLES_XLSX}")

# ---------------------------------------------------------------
# STILL UNRESOLVED AFTER ALL THREE PASSES
# (cross-referenced against skipped_id_style_codes if available, so
# genuine IDs aren't counted as "still needing work")
# ---------------------------------------------------------------
ABBR_TOKEN = re.compile(r"\b[A-Z]{2,8}\b")

entities_long = pd.concat([
    df[["expanded_source", DOI_COL]].rename(columns={"expanded_source": "entity"}),
    df[["expanded_target", DOI_COL]].rename(columns={"expanded_target": "entity"}),
]).dropna().drop_duplicates()

still_unresolved_rows = []
for _, row in entities_long.iterrows():
    entity, doi = str(row["entity"]), row[DOI_COL]
    for tok in ABBR_TOKEN.findall(entity):
        still_unresolved_rows.append({"doi": doi, "entity": entity, "undefined_token": tok})

still_unresolved_df = pd.DataFrame(still_unresolved_rows).drop_duplicates()
print(f"All-caps tokens remaining after all 3 passes: {still_unresolved_df['entity'].nunique() if len(still_unresolved_df) else 0} entities")

# ---------------------------------------------------------------
# SAVE AUDIT WORKBOOK: multi-sheet, for YOUR OWN review in Excel,
# not meant for re-upload (Claude.ai only accepts single-sheet xlsx)
# ---------------------------------------------------------------
summary_df = pd.DataFrame({
    "pass": ["1_per_paper", "2_global_dictionary", "3_verified_full_article", "4_human_reviewed"],
    "entries_available": [len(per_paper_df), len(GLOBAL_ABBREVIATIONS), len(verified_df), len(human_reviewed_df)],
})

with pd.ExcelWriter(OUTPUT_AUDIT_XLSX) as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    per_paper_df.to_excel(writer, sheet_name="pass1_per_paper_lookup", index=False)
    pd.DataFrame(list(GLOBAL_ABBREVIATIONS.items()), columns=["abbreviation", "full_form"]).to_excel(
        writer, sheet_name="pass2_global_lookup", index=False
    )
    verified_df.to_excel(writer, sheet_name="pass3_verified_lookup", index=False)
    human_reviewed_df.to_excel(writer, sheet_name="pass4_human_reviewed", index=False)
    still_unresolved_df.to_excel(writer, sheet_name="still_unresolved_after_all", index=False)

print(f"Saved audit workbook to {OUTPUT_AUDIT_XLSX}")
print("\nNext: run tier1_family_tagging.py / tier2_llm_code_extraction.py using")
print("expanded_source/expanded_target in place of source/target.")

Loaded 10324 triples
Pass 1 (per-paper definitions): 181 entries across 78 papers
Pass 2 (static global dictionary): 20 entries
Pass 3 (verified full-article resolutions): 82 entries across 52 papers (0 excluded as confirmed non-abbreviations)
Pass 4 (human-reviewed unverified rows): 32 entries across 26 papers (3 corrected by you, 11 nulls excluded)

Rows with source expanded: 733
Rows with target expanded: 199

Saved expanded triples to expanded_triples.xlsx
All-caps tokens remaining after all 3 passes: 107 entities
Saved audit workbook to consolidation_audit.xlsx

Next: run tier1_family_tagging.py / tier2_auto_merge_by_code.py using
expanded_source/expanded_target in place of source/target.
